In [16]:
from langgraph.graph  import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os
load_dotenv()
from typing import TypedDict

In [17]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

KeyError: 'GROQ_API_KEY'

In [ ]:
class studentInfo(TypedDict):
    studentInfo : str
    student_name : str
    student_roll_no : str
    student_course : str
    combined_output : str

In [ ]:
def name(state : studentInfo):
    responce = model.invoke(f"extract student name from this {state['studentInfo']}")
    return {"student_name" : responce.content}

In [ ]:
d = {
    "studentInfo": "Student name: Ada Lovelace, Course: Computer Science, Roll No: 42"
}

In [ ]:
def combined_output(state: studentInfo):
    responce = model.invoke(
        f"Combine these student details into one concise sentence: "
        f"name={state['student_name']}, "
        f"course={state['student_course']}, "
        f"roll number={state['student_roll_no']}"
    )

    return {"combined_output": responce.content}

In [ ]:
def course(state: studentInfo):
    responce = model.invoke(f"Extract the student's course from this text: {state['studentInfo']}")
    return {"student_course": responce.content}


def roll_no(state: studentInfo):
    responce = model.invoke(f"Extract the student's roll number from this text: {state['studentInfo']}")
    return {"student_roll_no": responce.content}


parallel_workflow = StateGraph(studentInfo)
parallel_workflow.add_node("student_name", name)
parallel_workflow.add_node("student_course", course)
parallel_workflow.add_node("student_roll_no", roll_no)
parallel_workflow.add_node("combined_output", combined_output)

parallel_workflow.add_edge(START, "student_name")
parallel_workflow.add_edge(START, "student_course")
parallel_workflow.add_edge(START, "student_roll_no")
parallel_workflow.add_edge(
    ["student_name", "student_course", "student_roll_no"],
    "combined_output"
)
parallel_workflow.add_edge("combined_output", END)
parallel_workflow = parallel_workflow.compile()

In [ ]:
result = parallel_workflow.invoke(d)
result["combined_output"]